https://huggingface.co/docs/transformers/generation_strategies

In [4]:
from transformers import pipeline
from transformers import set_seed

set_seed(42)

In [5]:
tg_pipeline = pipeline("text-generation", model="distilbert/distilgpt2", device=0)

tg_pipeline

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

TextGenerationPipeline: {'model': 'GPT2LMHeadModel', 'dtype': 'float32', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}

### Passing generation parameters as a `GenerationConfig`

The text-generation pipeline already hands `generate()` its own `generation_config`. Passing loose generation kwargs (like `pad_token_id`, `max_new_tokens`, `top_p`) *alongside* it triggers the deprecation warning about passing a config object and generation arguments together. The helper below bundles the parameters into a single `GenerationConfig` object so we pass exactly one. It also clears `max_length` whenever `max_new_tokens` is set, which removes the `max_new_tokens` vs `max_length` clash.

In [6]:
import copy
from transformers import GenerationConfig

def gen_config(pipe, **params):
    gc = copy.deepcopy(pipe.generation_config)

    for key, value in params.items():
        setattr(gc, key, value)

    if params.get("max_new_tokens") is not None:
        gc.max_length = None  # let max_new_tokens control length

    return gc

Note that there are many input parameters that can be used to configure and control text generation

https://huggingface.co/docs/transformers/main_classes/text_generation

In [7]:
text = "In a world where dreams become reality"

answer = tg_pipeline(text, generation_config=gen_config(tg_pipeline, pad_token_id=tg_pipeline.tokenizer.eos_token_id))

print(answer)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'In a world where dreams become reality, the only way to escape the obstacles is to escape the obstacles, and to have the courage to challenge the obstacles.'}]


In [8]:
text = "In a universe where every idea takes shape"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        min_new_tokens=100,
        max_new_tokens=500
    )
)

print(answer[0]['generated_text'])

[transformers] Both `min_new_tokens` (=100) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In a universe where every idea takes shape, the Universe is going to be a lot different from our own. But the possibilities of a universe are limitless.


The universe is going to be very different. It will have a very different future than the one we have at present, but it could be more like the one we have in the past.
So, how will we be able to take that reality and create a universe that we think is a more like the one we have in the past?
We can use the technologies we have today to create a universe that we think is totally different from what we have today. It's not like the universe is not in the past.
And I think that's what we should be able to find, because we know that there is something about the Universe that we shouldn't be thinking about because it is something that the Universe is not in.
Do you think that if we were to have a universe filled with human beings, it would be a lot different than the one we have today.
I think it is what we should be able to find.
So, wh

In [9]:
text = "In a universe where every idea takes shape"

answer = tg_pipeline(
    text,
    tokenizer = tg_pipeline.tokenizer,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100,
        stop_strings = ["evolve", "world"]
    )
)

print(answer[0]['generated_text'])

In a universe where every idea takes shape, the universe is shaped by an infinite number of galaxies, and the Earth is shaped by an infinite number of galaxies.




There are two basic principles for understanding the Universe . One of them is the notion that the universe is shaped by a set of galaxies. The other is the notion that the universe is shaped by a set of atoms, and that the universe is made of atoms.
This principle is called the "the Universe." This is what the universe is. The universe


In [10]:
text = "In a world sculpted by visionary ideas"

answers = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=80,
        num_return_sequences = 3
    )
)

for answer in answers:
    print(answer['generated_text'])
    print('-'*80)

In a world sculpted by visionary ideas, the first of its kind has been created by the creation of a new, new, new, and exciting, highly interactive computer-generated image.


















































--------------------------------------------------------------------------------
In a world sculpted by visionary ideas, it is a joy to look at the world around it.

The first is a fascinating tale of a young man named Dr. Walter White.
A young man is a scientist and a scientist who is a scientist and a scientist who is a scientist and a scientist who is a scientist. When he is young, Dr. Walter is recruited to work on a study of the universe.

--------------------------------------------------------------------------------
In a world sculpted by visionary ideas, an artistic process is a form of education, education, and opportunity.



(Thanks to Eric for the link)
--------------------------------------------------------------------------------


# ---- Only continue if sufficient time ----

## Decoding Strategies in Text Generation

Decoding strategies in text generation models refer to the methods used to generate text from the output probabilities produced by models like GPT, BART, or other transformer-based architectures. These strategies determine how the next word (or token) is selected during the text generation process, which can significantly influence the quality, coherence, and creativity of the generated text.

#### Greedy Search

How It Works: In Greedy Search, the model selects the token with the highest probability at each step. This approach is straightforward but often leads to suboptimal results because it doesn’t consider the long-term implications of each choice.



In [ ]:
text = "I went to the office one day"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100
    )
)

print(answer[0]['generated_text'])

I went to the office one day and that's when I saw that one thing. That's when the whole thing really started to get crazy.



"I felt that I was in a good mood."


#### Beam Search
How It Works: Beam Search keeps track of multiple possible sequences (beams) at each step, rather than just the single best one. It explores a fixed number of top candidates (beam width) and expands them simultaneously, eventually selecting the sequence with the highest overall score.

In [ ]:
text = "I went to the office one day"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100,
        num_beams=4
    )
)

print(answer[0]['generated_text'])

I went to the office one day and said, 'I don't want to go to the office. I don't want to go to the office.'"














































































In [ ]:
text = "I went to the office one day"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100,
        num_beams=5,
        no_repeat_ngram_size=2
    )
)

print(answer[0]['generated_text'])

I went to the office one day and said, 'I don't know what to do, but I'm going to try to get a job.' I said to myself: 'Well, I can't do that. I've got to go back to work. But I'll do my best to make sure that I get the job done.' And then I got a call from a friend of mine who said that she wanted to come back. And I told her that it was a good idea, and she said she'd like to


In [ ]:
text = "I went to the office one day"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100,
        num_beams=5,
        no_repeat_ngram_size=4
    )
)

print(answer[0]['generated_text'])

I went to the office one day, and he said, 'I don't know what I'm going to do, but I'll do it.' He said, 'You know, I'm not going to do it. I don't want to do it.'



He said: 'I'm going to take care of myself and I'll do everything I can to make sure I'm doing the right thing.'
He added: 'I've got a lot of work to do, and I've got to keep


In [ ]:
text = "I went to the office one day"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=200,
        num_beams=5,
        repetition_penalty=2.0  # Default is 1.0 which means no penalty
    )
)

print(answer[0]['generated_text'])

I went to the office one day and I said, 'What are you doing?' "































































































































































































### Multinomial Sampling
As opposed to greedy search that always chooses a token with the highest probability as the next token, multinomial sampling (also called ancestral sampling) randomly selects the next token based on the probability distribution over the entire vocabulary given by the model. Every token with a non-zero probability has a chance of being selected, thus reducing the risk of repetition.

In [ ]:
text = "I went to the office one day"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100,
        do_sample=True,
        num_beams=1
    )
)

print(answer[0]['generated_text'])

I went to the office one day and asked why they were in the office.



“I asked if they were in the office and they said, 'I was in the office.'  
“I told them they were in the office and they said, 'I am in the office.'
“I asked if they were in the office.  
“They said, 'Well, we are in the office."
“They said, 'I have been in the office


#### Temperature:

A hyperparameter that controls the randomness of predictions, with lower values leading to more deterministic outputs and higher values introducing more variability.

#### Top-K Sampling:

A decoding strategy that restricts token selection to the top K most probable options, introducing controlled randomness by only considering a fixed number of high-probability tokens.

#### Top-p (Nucleus) Sampling:

A decoding strategy that selects tokens from the smallest set whose cumulative probability exceeds a threshold p, dynamically adjusting the number of considered tokens based on the context.

In [ ]:
text = "In a world sculpted by visionary ideas"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.1,  # Default 1.0
        top_k=0,
        top_p=0
    )
)

print(answer[0]['generated_text'])

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In a world sculpted by visionary ideas, the world of the artist is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art


In [ ]:
text = "In a world sculpted by visionary ideas"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=200,
        do_sample=True,
        temperature=1.0,
        top_k=0,
        top_p=0
    )
)


print(answer[0]['generated_text'])

In a world sculpted by visionary ideas, the world of the artist is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art


In [ ]:
text = "In a world sculpted by visionary ideas"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=200,
        do_sample=True,
        temperature=1.5,
        top_k=0,
        top_p=0
    )
)


print(answer[0]['generated_text'])

In a world sculpted by visionary ideas, the world of the artist is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art. It is a world of art


In [ ]:
text = "In a world sculpted by visionary ideas"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100,
        do_sample=True,
        top_k=10  # Default value is 50
    )
)

print(answer[0]['generated_text'])

In a world sculpted by visionary ideas like the first to be exhibited at the Museum of Art in New York, the exhibit was inspired by the works of Leonardo da Vinci, Leonardo da Vinci, and others, including the famous Leonardo da Vinci painting.


The exhibition is a collaboration between the Museum of Art and the National Museum of Art in New York's Central Park. The exhibit includes a series of paintings from the late Leonardo da Vinci. The exhibition was commissioned by the Smithsonian Institution's National Museum of Art,


In [ ]:
text = "In a world sculpted by visionary ideas"

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100,
        do_sample=True,
        top_k=100
    )
)

print(answer[0]['generated_text'])

In a world sculpted by visionary ideas, the art historian Robert Leibniz has become one of the most influential people to develop modern art in a century.




The artist, who has since become a leading figure in the art world, was commissioned by the National Geographic magazine, which in 2010 launched an online magazine called La Jolla.
Brigitte, who is also the co-creator of the first of the company's current project, said he was inspired by Leibniz's contribution to creating


In [ ]:
text = "In a world sculpted by visionary ideas"

print(text)

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100,
        do_sample=True,
        top_p=0.5  # Default value is 1
    )
)

print(answer[0]['generated_text'])

In a world sculpted by visionary ideas
In a world sculpted by visionary ideas, the artist has been able to create a unique and unique style of art that is unique to the world.
















































































In [ ]:
text = "In a world sculpted by visionary ideas"

print(text)

answer = tg_pipeline(
    text,
    generation_config=gen_config(
        tg_pipeline,
        pad_token_id=tg_pipeline.tokenizer.eos_token_id,
        max_new_tokens=100,
        do_sample=True,
        top_p=0.80
    )
)

print(answer[0]['generated_text'])

In a world sculpted by visionary ideas
In a world sculpted by visionary ideas, the World of Modern Art has a unique vision for the art of art, and we're proud to present the world's first of its kind.



The World of Modern Art is a collaboration between the American Museum of Art and the Smithsonian's National Museum of Art, the Smithsonian's National Museum of Art, and the Smithsonian's National Museum of Art.

The World of Modern Art is a collaboration between the American Museum of Art and the Smithsonian's National Museum of Art.



### Streaming support

You can use the TextStreamer class to stream the output of generate() into your screen, one word at a time

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
inputs = tokenizer(["An increasing sequence: one,"], return_tensors="pt")

streamer = TextStreamer(tokenizer)

In [ ]:
_ = model.generate(**inputs, streamer=streamer, max_new_tokens=30)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


An increasing sequence: one, two, three, four, five, six, seven, eight, nine, ten, eleven, twelve, thirteen, fourteen, fifteen, sixteen,
